# 📏 Notebook 05 — Production Baseline Benchmarking

**Project:** Demand Forecasting — Corporación Favorita Grocery Sales  
**Author:** Senior ML Engineer  
**Purpose:** Establish baseline benchmark scores before advanced model training (Notebook 06)  
**Environment:** Google Colab / Local (VS Code, Jupyter)  

---

## 🎯 Objective

This notebook is **NOT** the final model. It serves three purposes:

1. **Validate** that the Feature Store from Notebook 04 produces learnable signals.
2. **Establish** baseline benchmark scores that every advanced model must beat.
3. **Build** a professional evaluation report with metrics, charts, and exported artifacts.

> **Notebook 06** will contain: LightGBM, CatBoost, XGBoost, Hyperparameter Tuning.

## ⚡ Memory & Execution Strategy

| Design Decision | Implementation |
|----------------|----------------|
| **Chronological Subsetting** | `df.tail(10_000_000)` — recent 10M rows only |
| **Zero Random Sampling** | Strict chronological order preserved everywhere |
| **Zero Dtype Recasting** | Feature Store already optimized (`float32`, `uint8`, `category`) |
| **Sequential Training** | Train → Evaluate → Save → Delete → GC after every model |
| **Error Resilience** | Every model wrapped in `try/except` — one failure never crashes the notebook |
| **RAM Monitoring** | Memory usage reported after every major operation |

## 🏗️ Pipeline Architecture

```
feature_store.parquet (Notebook 04 Output — 86.9M rows, ~5.8 GB on disk)
        ↓
Chronological Tail Subsetting → Recent 10,000,000 Rows (~85% RAM Reduction)
        ↓
Automatic Feature Detection (numeric, categorical, boolean, leakage columns)
        ↓
Strict Out-of-Time Train/Val Split (Val: 2017-08-01 → end)
        ↓
┌──────────────────────────────────────────────────────────────────────────┐
│ SEQUENTIAL BASELINE ENGINE (Train → Eval → Save → Delete → GC)         │
│                                                                          │
│   1. Mean Baseline         ──► Evaluate ──► Log ──► GC                  │
│   2. Median Baseline       ──► Evaluate ──► Log ──► GC                  │
│   3. Seasonal Naive (t-7)  ──► Evaluate ──► Log ──► GC                  │
│   4. Linear Regression     ──► Evaluate ──► Save ──► GC                 │
│   5. Ridge Regression      ──► Evaluate ──► Save ──► GC                 │
│   6. Lasso Regression      ──► Evaluate ──► Save ──► GC                 │
│   7. ElasticNet            ──► Evaluate ──► Save ──► GC                 │
│   8. Random Forest         ──► Evaluate ──► Save ──► GC                 │
└──────────────────────────────────────────────────────────────────────────┘
        ↓
Export: baseline_results.csv, metrics.json, benchmark_summary.csv, feature_list.txt
        ↓
Diagnostic Visualization Suite (5 High-DPI Charts)
```

## 📊 Evaluation Metrics

| Metric | Formula | Purpose |
|--------|---------|--------|
| **RMSLE** | $\sqrt{\frac{1}{n}\sum(\log(1+\hat{y}) - \log(1+y))^2}$ | Primary competition metric |
| **RMSE** | $\sqrt{\frac{1}{n}\sum(\hat{y} - y)^2}$ | Scale-sensitive error |
| **MAE** | $\frac{1}{n}\sum|\hat{y} - y|$ | Interpretable absolute error |
| **MAPE** | $\frac{100}{n}\sum|\frac{\hat{y} - y}{y}|$ | Percentage error (excludes zeros) |
| **R²** | $1 - \frac{SS_{res}}{SS_{tot}}$ | Variance explained |

| Specification | Detail |
|---------------|--------|
| **Inputs** | `01_Dataset/features/feature_store.parquet` |
| **Outputs** | `baseline_results.csv`, `metrics.json`, `benchmark_summary.csv`, Models, Charts |
| **Previous Notebook** | `04_feature_engineering.ipynb` |
| **Next Notebook** | `06_model_training.ipynb` |

---


## 1️⃣ Environment Setup & Project Bootstrap

Detects execution environment (Google Colab vs Local) and configures project paths.


In [ ]:
# ============================================================
# 1. Environment Bootstrap (Colab + Local)
# ============================================================
import os, sys
from pathlib import Path

# 1. Google Colab: Mount Drive
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

# 2. Find project root (checks Drive paths + local paths)
POSSIBLE_ROOTS = [
    Path('/content/drive/MyDrive/Demand-Forecasting-System'),
    Path('/content/drive/MyDrive/NTI/Demand-Forecasting-System'),
    Path('/content/drive/MyDrive/Colab Notebooks/Demand-Forecasting-System'),
    Path.cwd(),
    Path.cwd().parent,
]

PROJECT_ROOT = None
for p in POSSIBLE_ROOTS:
    if p.exists() and (p / 'config.py').exists():
        PROJECT_ROOT = p.resolve()
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        '❌ Project root with config.py not found.\n'
        'Colab: Make sure the folder is in your Google Drive.\n'
        'Local: Run the notebook from inside the project directory.'
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

ENV = 'Google Colab' if 'google.colab' in sys.modules else 'Local'
print('=' * 60)
print(f'📁 Project Root : {PROJECT_ROOT}')
print(f'📂 Working Dir  : {os.getcwd()}')
print(f'🖥️  Runtime      : {ENV}')
print('✅ Bootstrap OK')
print('=' * 60)


## 2️⃣ Imports & Configuration

Imports all required libraries, configures Matplotlib aesthetics, and sets up output directories.


In [ ]:
# ============================================================
# 2. Imports & Configuration
# ============================================================
import time, gc, json, warnings, psutil
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import config, utils

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 80)
pd.set_option('display.float_format', lambda x: '%.4f' % x)

# ── Matplotlib Presentation Theme ──
plt.rcParams.update({
    'font.sans-serif': 'DejaVu Sans',
    'axes.edgecolor': '#cccccc',
    'axes.linewidth': 1.0,
    'grid.color': '#eeeeee',
    'grid.linestyle': '--',
    'figure.dpi': 150,
    'axes.titleweight': 'bold',
    'axes.titlesize': 13
})

# ── Output Directories ──
OUTPUT_DIR = Path(config.PROJECT_ROOT) / 'output' / '05_baseline_models'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR = OUTPUT_DIR / 'baseline_models'
MODELS_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR = OUTPUT_DIR / 'plots'
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

# ── RAM Monitoring Helper ──
def log_ram(label=''):
    """Print current process RSS memory usage."""
    rss = psutil.Process().memory_info().rss / (1024**3)
    print(f'🧠 RAM [{label}]: {rss:.2f} GB')

# ── Figure Save Helper ──
def save_fig(fig, filename):
    """Save figure to plots directory, display, then close to free memory."""
    filepath = PLOTS_DIR / filename
    fig.savefig(filepath, dpi=200, bbox_inches='tight', facecolor='white')
    print(f'💾 Saved → {filepath}')
    plt.show()
    plt.close(fig)  # Free figure memory

NOTEBOOK_START = time.time()
log_ram('After Imports')
print('✅ Configuration complete.')


## 3️⃣ Load Feature Store

### 🎯 Purpose
Load the Feature Store generated by Notebook 04 via `utils.load_feature_store()`.

### ⚡ Memory Strategy
- Direct Parquet load — **zero dtype recasting** (Feature Store is already optimized).
- Immediately take `tail(BASELINE_ROWS)` to reduce RAM by ~85%.
- Delete the full DataFrame reference immediately after subsetting.


In [ ]:
# ============================================================
# 3. Load Feature Store & Chronological Subsetting
# ============================================================
BASELINE_ROWS = 5_000_000  # ⚠️ Adjust based on available RAM

print('⏳ Loading feature_store.parquet...')
t0 = time.time()
df_full = utils.load_feature_store(verbose=True)
load_time = time.time() - t0

full_rows = len(df_full)
full_cols = len(df_full.columns)
full_mem  = df_full.memory_usage(deep=True).sum() / 1e9
log_ram('After Full Load')

# ── Chronological Subsetting (tail — NO random sampling) ──
print(f'\n✂️ Selecting most recent {BASELINE_ROWS:,} rows chronologically...')
df = df_full.tail(BASELINE_ROWS).copy()
del df_full
gc.collect()

subset_mem = df.memory_usage(deep=True).sum() / 1e9
log_ram('After Subset + GC')

print(f'\n{"=" * 70}')
print('📊 DATASET SUMMARY')
print(f'{"=" * 70}')
print(f'   Full Feature Store    : {full_rows:>15,} rows | {full_cols} cols | {full_mem:.2f} GB')
print(f'   Baseline Subset       : {len(df):>15,} rows | {full_cols} cols | {subset_mem:.2f} GB')
print(f'   RAM Reduction         : {(1 - subset_mem / full_mem) * 100:.1f}%')
print(f'   Date Range            : {df["date"].min()} → {df["date"].max()}')
print(f'   Load Time             : {load_time:.2f}s')
print(f'{"=" * 70}')


## 4️⃣ Automatic Feature Detection & Leakage Prevention

### 🎯 Purpose
Automatically classify every column by type (target, date, ID, categorical, boolean, numeric) and exclude leakage/non-feature columns.

### 🚫 Excluded Columns
| Column | Reason |
|--------|--------|
| `date` | Temporal identifier — not a predictive feature |
| `id` | Row identifier — no predictive value |
| `unit_sales` | Target variable — must be separated |
| String categoricals | Already frequency-encoded in Notebook 04 |


In [ ]:
# ============================================================
# 4. Automatic Feature Detection & Isolation
# ============================================================
TARGET_COL = 'unit_sales'

# Columns to exclude from feature matrix
LEAKAGE_COLS = ['date', 'id', 'unit_sales']
# String/category metadata columns (already encoded as frequency features)
META_COLS = ['family', 'city', 'state', 'type', 'holiday_type', 'holiday_locale']
IGNORED_COLS = set(LEAKAGE_COLS + META_COLS)

# ── Automatic Column Classification ──
all_cols     = df.columns.tolist()
num_cols     = df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols     = df.select_dtypes(include=['category', 'object']).columns.tolist()
bool_cols    = df.select_dtypes(include=['bool']).columns.tolist()
feature_cols = [c for c in num_cols if c not in IGNORED_COLS]

print(f'{"=" * 70}')
print('🔍 AUTOMATIC FEATURE DETECTION REPORT')
print(f'{"=" * 70}')
print(f'   🎯 Target Column      : {TARGET_COL}')
print(f'   📊 Numeric Columns     : {len(num_cols)}')
print(f'   📝 Categorical Columns : {len(cat_cols)}')
print(f'   ✅ Boolean Columns     : {len(bool_cols)}')
print(f'   🚫 Excluded Columns    : {sorted(IGNORED_COLS & set(all_cols))}')
print(f'   ✅ Selected Features   : {len(feature_cols)}')
print(f'{"=" * 70}')

# ── Save feature list ──
feat_path = OUTPUT_DIR / 'feature_list.txt'
with open(feat_path, 'w', encoding='utf-8') as f:
    f.write('\n'.join(feature_cols))
print(f'💾 Feature list saved → {feat_path}')


## 5️⃣ Chronological Out-of-Time Train / Validation Split

### 🎯 Purpose
Split the dataset using a **strict temporal cutoff** to prevent data leakage.

### ⚠️ Why Random Split is Forbidden
In time-series forecasting, random splits leak future information into training data, producing
over-optimistic scores that **fail catastrophically in production**.

| Set | Condition | Purpose |
|-----|-----------|--------|
| **Training** | $t < \text{2017-08-01}$ | Learn historical patterns |
| **Validation** | $t \geq \text{2017-08-01}$ | Simulate future prediction (matches Kaggle test horizon) |


In [ ]:
# ============================================================
# 5. Chronological Train / Validation Split
# ============================================================
VAL_START = '2017-08-01'

df['date'] = pd.to_datetime(df['date'])
train_mask = df['date'] < VAL_START
val_mask   = df['date'] >= VAL_START

# ── Extract feature matrices as float32 NumPy arrays ──
X_train = df.loc[train_mask, feature_cols].fillna(0).values.astype(np.float32)
y_train = df.loc[train_mask, TARGET_COL].values.astype(np.float32)

X_val = df.loc[val_mask, feature_cols].fillna(0).values.astype(np.float32)
y_val = df.loc[val_mask, TARGET_COL].values.astype(np.float32)

# Keep sales_lag_7 for Seasonal Naive before dropping df reference
snaive_vals = df.loc[val_mask, 'sales_lag_7'].fillna(0).values.astype(np.float32) if 'sales_lag_7' in df.columns else None

# ── Report ──
train_dates = df.loc[train_mask, 'date']
val_dates   = df.loc[val_mask, 'date']

print(f'{"=" * 70}')
print('📅 CHRONOLOGICAL TRAIN / VALIDATION SPLIT')
print(f'{"=" * 70}')
print(f'   Training Period   : {train_dates.min().date()} → {train_dates.max().date()}')
print(f'   Training Rows     : {len(X_train):,}')
print(f'   Training RAM      : {X_train.nbytes / 1e9:.2f} GB')
print(f'   Validation Period : {val_dates.min().date()} → {val_dates.max().date()}')
print(f'   Validation Rows   : {len(X_val):,}')
print(f'   Validation RAM    : {X_val.nbytes / 1e9:.2f} GB')
print(f'   Features Used     : {X_train.shape[1]}')
print(f'{"=" * 70}')

del train_dates, val_dates
gc.collect()
log_ram('After Split')


### 📊 5.1 Visualization: Temporal Train / Validation Split


In [ ]:
# ── 5.1 Temporal Split Visualization ──
daily_agg = df.groupby('date')[TARGET_COL].sum().reset_index()

fig, ax = plt.subplots(figsize=(14, 4))
t_mask = daily_agg['date'] < VAL_START
ax.plot(daily_agg.loc[t_mask, 'date'], daily_agg.loc[t_mask, TARGET_COL],
        label='Training Period', color='#3498db', linewidth=1.5)
ax.plot(daily_agg.loc[~t_mask, 'date'], daily_agg.loc[~t_mask, TARGET_COL],
        label='Validation Period', color='#e74c3c', linewidth=2)
ax.axvline(pd.Timestamp(VAL_START), color='red', linestyle='--', linewidth=2,
           label=f'Split Point ({VAL_START})')
ax.set_title('Chronological Out-of-Time Train / Validation Split', fontweight='bold')
ax.set_ylabel('Total Daily Sales')
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)
save_fig(fig, '00_chronological_train_val_split.png')

del daily_agg
gc.collect()


## 6️⃣ Evaluation Framework & Metric Logger

### 🎯 Purpose
Central evaluation function that computes all metrics, saves models, logs results, and manages memory.

### 📊 Metrics Computed
| Metric | Description |
|--------|------------|
| RMSLE | Root Mean Squared Log Error (primary competition metric) |
| RMSE | Root Mean Squared Error |
| MAE | Mean Absolute Error |
| MAPE | Mean Absolute Percentage Error |
| R² | Coefficient of Determination |
| Train Time | Seconds to fit the model |
| Pred Time | Seconds to generate predictions |
| Model Size | Saved model file size in MB |


In [ ]:
# ============================================================
# 6. Evaluation Framework & Metric Logger
# ============================================================
baseline_results = []

def safe_mape(y_true, y_pred, eps=1.0):
    """Compute MAPE, excluding near-zero actuals to avoid division by zero."""
    mask = np.abs(y_true) > eps
    if mask.sum() == 0:
        return 0.0
    return float(np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100)

def evaluate_model(model_name, y_true, y_pred, train_time=0.0, pred_time=0.0,
                   model_obj=None, filename=None):
    """Evaluate, log, optionally save model, and return metrics dict."""
    # Clip negatives (sales cannot be negative)
    y_t = np.clip(y_true, 0, None)
    y_p = np.clip(y_pred, 0, None)

    rmse  = float(np.sqrt(mean_squared_error(y_t, y_p)))
    mae   = float(mean_absolute_error(y_t, y_p))
    rmsle = float(np.sqrt(mean_squared_error(np.log1p(y_t), np.log1p(y_p))))
    r2    = float(r2_score(y_t, y_p))
    mape  = safe_mape(y_t, y_p)

    # Save model if provided
    model_size_mb = 0.0
    if model_obj is not None and filename is not None:
        save_path = MODELS_DIR / filename
        joblib.dump(model_obj, save_path, compress=3)
        model_size_mb = float(save_path.stat().st_size / (1024 * 1024))
        print(f'   💾 Model saved → {save_path.name} ({model_size_mb:.2f} MB)')

    res = {
        'Model': model_name,
        'RMSLE': round(rmsle, 4),
        'RMSE': round(rmse, 2),
        'MAE': round(mae, 2),
        'MAPE': round(mape, 2),
        'R2_Score': round(r2, 4),
        'Train_Time_s': round(train_time, 2),
        'Pred_Time_s': round(pred_time, 4),
        'Model_Size_MB': round(model_size_mb, 3)
    }
    baseline_results.append(res)

    print(f'   📊 {model_name}')
    print(f'      RMSLE: {rmsle:.4f} | RMSE: {rmse:.2f} | MAE: {mae:.2f} | MAPE: {mape:.2f}% | R²: {r2:.4f}')
    print(f'      Train: {train_time:.2f}s | Predict: {pred_time:.4f}s')
    log_ram(f'After {model_name}')
    return res

print('✅ Evaluation framework ready.')


## 7️⃣ Baseline Model Training & Evaluation

### 🏗️ Execution Protocol
Each model follows the same strict sequence:

1. **Train** the model (timed)
2. **Predict** on validation set (timed)
3. **Evaluate** all metrics
4. **Save** model to disk (if applicable)
5. **Delete** model object and predictions from RAM
6. **Garbage Collect** to reclaim memory

> ⚠️ Every model is wrapped in `try/except`. If one model fails, the notebook continues execution.


### 7️⃣.1 Baseline 1 — Global Mean

Predicts the global average of training sales: $\hat{y} = \bar{y}_{train}$  
Establishes the **absolute simplest lower bound**.


In [ ]:
# ── Baseline 1: Global Mean ──
try:
    t0 = time.time()
    mean_val = float(np.mean(y_train))
    train_time = time.time() - t0

    t0 = time.time()
    y_pred = np.full(len(y_val), mean_val, dtype=np.float32)
    pred_time = time.time() - t0

    evaluate_model('Global Mean', y_val, y_pred, train_time, pred_time)
    del y_pred; gc.collect()
except Exception as e:
    print(f'❌ Global Mean failed: {e}')


### 7️⃣.2 Baseline 2 — Global Median

Predicts the global median: $\hat{y} = \text{Median}(y_{train})$  
More robust to extreme promotional sales spikes than the mean.


In [ ]:
# ── Baseline 2: Global Median ──
try:
    t0 = time.time()
    median_val = float(np.median(y_train))
    train_time = time.time() - t0

    t0 = time.time()
    y_pred = np.full(len(y_val), median_val, dtype=np.float32)
    pred_time = time.time() - t0

    evaluate_model('Global Median', y_val, y_pred, train_time, pred_time)
    del y_pred; gc.collect()
except Exception as e:
    print(f'❌ Global Median failed: {e}')


### 7️⃣.3 Baseline 3 — Seasonal Naive ($t-7$)

Predicts sales equal to the same day last week: $\hat{y}_t = y_{t-7}$  
Captures weekly seasonality patterns (e.g., weekday vs weekend demand cycles).


In [ ]:
# ── Baseline 3: Seasonal Naive (t-7) ──
try:
    train_time = 0.0  # No training needed

    t0 = time.time()
    if snaive_vals is not None:
        y_pred = snaive_vals.copy()
    else:
        # Fallback: use global mean if sales_lag_7 not available
        y_pred = np.full(len(y_val), float(np.mean(y_train)), dtype=np.float32)
        print('   ⚠️ sales_lag_7 not found, falling back to mean.')
    pred_time = time.time() - t0

    evaluate_model('Seasonal Naive (t-7)', y_val, y_pred, train_time, pred_time)
    del y_pred; gc.collect()
except Exception as e:
    print(f'❌ Seasonal Naive failed: {e}')


### 7️⃣.4 Baseline 4 — Linear Regression (OLS)

Unregularized ordinary least squares trained on $\log(1+y)$ transformed target.  
Establishes the parametric linear baseline.


In [ ]:
# ── Baseline 4: Linear Regression ──
try:
    y_train_log = np.log1p(np.clip(y_train, 0, None))

    t0 = time.time()
    model = LinearRegression()
    model.fit(X_train, y_train_log)
    train_time = time.time() - t0

    t0 = time.time()
    y_pred = np.expm1(model.predict(X_val)).astype(np.float32)
    pred_time = time.time() - t0

    evaluate_model('Linear Regression', y_val, y_pred, train_time, pred_time,
                   model, 'baseline_linear_regression.joblib')
    del model, y_pred; gc.collect()
except Exception as e:
    print(f'❌ Linear Regression failed: {e}')


### 7️⃣.5 Baseline 5 — Ridge Regression ($L_2$)

Adds $L_2$ penalty ($\alpha = 1.0$) to stabilize coefficients against multicollinearity
in highly correlated lag and rolling window features.


In [ ]:
# ── Baseline 5: Ridge Regression ──
try:
    y_train_log = np.log1p(np.clip(y_train, 0, None))

    t0 = time.time()
    model = Ridge(alpha=1.0, random_state=42)
    model.fit(X_train, y_train_log)
    train_time = time.time() - t0

    t0 = time.time()
    y_pred = np.expm1(model.predict(X_val)).astype(np.float32)
    pred_time = time.time() - t0

    evaluate_model('Ridge Regression', y_val, y_pred, train_time, pred_time,
                   model, 'baseline_ridge.joblib')
    del model, y_pred; gc.collect()
except Exception as e:
    print(f'❌ Ridge Regression failed: {e}')


### 7️⃣.6 Baseline 6 — Lasso Regression ($L_1$)

Adds $L_1$ penalty ($\alpha = 0.01$) to enforce coefficient sparsity,
automatically zeroing out weak or redundant features.


In [ ]:
# ── Baseline 6: Lasso Regression ──
try:
    y_train_log = np.log1p(np.clip(y_train, 0, None))

    t0 = time.time()
    model = Lasso(alpha=0.01, random_state=42, max_iter=2000)
    model.fit(X_train, y_train_log)
    train_time = time.time() - t0

    t0 = time.time()
    y_pred = np.expm1(model.predict(X_val)).astype(np.float32)
    pred_time = time.time() - t0

    evaluate_model('Lasso Regression', y_val, y_pred, train_time, pred_time,
                   model, 'baseline_lasso.joblib')
    del model, y_pred; gc.collect()
except Exception as e:
    print(f'❌ Lasso Regression failed: {e}')


### 7️⃣.7 Baseline 7 — ElasticNet ($L_1 + L_2$)

Combines $L_1$ sparsity and $L_2$ grouping penalties ($\alpha = 0.01, l_1\_ratio = 0.5$).  
Effective when features are correlated in groups (e.g., multiple rolling window features).


In [ ]:
# ── Baseline 7: ElasticNet ──
try:
    y_train_log = np.log1p(np.clip(y_train, 0, None))

    t0 = time.time()
    model = ElasticNet(alpha=0.01, l1_ratio=0.5, random_state=42, max_iter=2000)
    model.fit(X_train, y_train_log)
    train_time = time.time() - t0

    t0 = time.time()
    y_pred = np.expm1(model.predict(X_val)).astype(np.float32)
    pred_time = time.time() - t0

    evaluate_model('ElasticNet', y_val, y_pred, train_time, pred_time,
                   model, 'baseline_elasticnet.joblib')
    del model, y_pred; gc.collect()
except Exception as e:
    print(f'❌ ElasticNet failed: {e}')


### 7️⃣.8 Baseline 8 — Random Forest (Small)

Non-linear tree ensemble baseline lower bound.  
Deliberately kept small ($n\_estimators = 50$, $max\_depth = 10$) for RAM safety and speed.


In [ ]:
# ── Baseline 8: Random Forest (Lightweight) ──
try:
    y_train_log = np.log1p(np.clip(y_train, 0, None))

    t0 = time.time()
    model = RandomForestRegressor(
        n_estimators=50, max_depth=10,
        max_features='sqrt', min_samples_leaf=50,
        n_jobs=-1, random_state=42
    )
    model.fit(X_train, y_train_log)
    train_time = time.time() - t0

    t0 = time.time()
    y_pred = np.expm1(model.predict(X_val)).astype(np.float32)
    pred_time = time.time() - t0

    evaluate_model('Random Forest', y_val, y_pred, train_time, pred_time,
                   model, 'baseline_random_forest.joblib')
    del model, y_pred; gc.collect()
except Exception as e:
    print(f'❌ Random Forest failed: {e}')

# Clean up log-transformed target
if 'y_train_log' in dir():
    del y_train_log
gc.collect()
log_ram('After All Models')


## 8️⃣ Master Benchmark Results & Export

### 🎯 Purpose
Compile all baseline metrics into a master results table sorted by **RMSLE** (primary competition metric).

### 📦 Exported Artifacts
| File | Description |
|------|------------|
| `baseline_results.csv` | Full results table with all metrics |
| `metrics.json` | Machine-readable metrics dictionary |
| `benchmark_summary.csv` | Condensed summary for quick comparison |
| `feature_list.txt` | List of features used for training |


In [ ]:
# ============================================================
# 8. Master Benchmark Results & Export
# ============================================================
if not baseline_results:
    print('⚠️ No models completed successfully. Skipping export.')
else:
    df_results = pd.DataFrame(baseline_results).sort_values('RMSLE').reset_index(drop=True)

    print(f'{"=" * 90}')
    print('🏆 MASTER BASELINE BENCHMARK RESULTS')
    print(f'{"=" * 90}')
    print(df_results.to_string(index=False))
    print(f'{"=" * 90}')

    # ── Export baseline_results.csv ──
    csv_path = OUTPUT_DIR / 'baseline_results.csv'
    df_results.to_csv(csv_path, index=False)
    print(f'\n💾 CSV saved → {csv_path}')

    # ── Export metrics.json ──
    json_path = OUTPUT_DIR / 'metrics.json'
    metrics_dict = {row['Model']: row for row in baseline_results}
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(metrics_dict, f, indent=2, ensure_ascii=False)
    print(f'💾 JSON saved → {json_path}')

    # ── Export benchmark_summary.csv ──
    summary = df_results[['Model', 'RMSLE', 'RMSE', 'MAE', 'R2_Score']].copy()
    summary_path = OUTPUT_DIR / 'benchmark_summary.csv'
    summary.to_csv(summary_path, index=False)
    print(f'💾 Summary saved → {summary_path}')
    del summary

    # ── Identify best model ──
    best = df_results.iloc[0]
    print(f'\n🏆 Best Baseline: {best["Model"]} (RMSLE: {best["RMSLE"]:.4f})')


## 9️⃣ Diagnostic Visualization Suite

### 🎯 Purpose
Generate presentation-quality diagnostic charts for the graduation report.

All figures are saved at **200 DPI** to `output/05_baseline_models/plots/` and closed immediately to free memory.


### 📊 9.1 Model Error Metrics Comparison


In [ ]:
# ── 9.1 Model Error Metrics Comparison (RMSLE, RMSE, MAE, R²) ──
if baseline_results:
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(df_results)))

    # RMSLE
    df_results.plot(x='Model', y='RMSLE', kind='barh', ax=axes[0, 0],
                    color=colors, edgecolor='black', legend=False)
    axes[0, 0].set_title('RMSLE (Lower = Better)', fontweight='bold')
    for p in axes[0, 0].patches:
        axes[0, 0].text(p.get_width() + 0.005, p.get_y() + p.get_height()/2,
                        f'{p.get_width():.4f}', va='center', fontsize=9)

    # RMSE
    df_results.plot(x='Model', y='RMSE', kind='barh', ax=axes[0, 1],
                    color=colors, edgecolor='black', legend=False)
    axes[0, 1].set_title('RMSE (Lower = Better)', fontweight='bold')
    for p in axes[0, 1].patches:
        axes[0, 1].text(p.get_width() + 0.5, p.get_y() + p.get_height()/2,
                        f'{p.get_width():.1f}', va='center', fontsize=9)

    # MAE
    df_results.plot(x='Model', y='MAE', kind='barh', ax=axes[1, 0],
                    color=colors, edgecolor='black', legend=False)
    axes[1, 0].set_title('MAE (Lower = Better)', fontweight='bold')
    for p in axes[1, 0].patches:
        axes[1, 0].text(p.get_width() + 0.2, p.get_y() + p.get_height()/2,
                        f'{p.get_width():.1f}', va='center', fontsize=9)

    # R²
    df_results.plot(x='Model', y='R2_Score', kind='barh', ax=axes[1, 1],
                    color=colors, edgecolor='black', legend=False)
    axes[1, 1].set_title('R² Score (Higher = Better)', fontweight='bold')
    for p in axes[1, 1].patches:
        axes[1, 1].text(p.get_width() + 0.01, p.get_y() + p.get_height()/2,
                        f'{p.get_width():.4f}', va='center', fontsize=9)

    plt.suptitle('Baseline Model Error Metrics Comparison', fontsize=15, fontweight='bold', y=1.02)
    plt.tight_layout()
    save_fig(fig, '01_model_metrics_comparison.png')


### 📊 9.2 Operational Metrics Comparison


In [ ]:
# ── 9.2 Operational Metrics (Training Time, Prediction Time, Model Size) ──
if baseline_results:
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    df_results.plot(x='Model', y='Train_Time_s', kind='barh', ax=axes[0],
                    color='#e74c3c', edgecolor='black', legend=False)
    axes[0].set_title('Training Time (Seconds)', fontweight='bold')

    df_results.plot(x='Model', y='Pred_Time_s', kind='barh', ax=axes[1],
                    color='#f39c12', edgecolor='black', legend=False)
    axes[1].set_title('Prediction Time (Seconds)', fontweight='bold')

    df_results.plot(x='Model', y='Model_Size_MB', kind='barh', ax=axes[2],
                    color='#3498db', edgecolor='black', legend=False)
    axes[2].set_title('Saved Model Size (MB)', fontweight='bold')

    plt.suptitle('Operational Performance Comparison', fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    save_fig(fig, '02_operational_metrics_comparison.png')


### 📊 9.3 MAPE Comparison


In [ ]:
# ── 9.3 MAPE Comparison ──
if baseline_results:
    fig, ax = plt.subplots(figsize=(10, 5))
    colors = plt.cm.plasma(np.linspace(0.2, 0.8, len(df_results)))
    bars = ax.barh(df_results['Model'], df_results['MAPE'], color=colors, edgecolor='black')
    for bar in bars:
        ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
                f'{bar.get_width():.1f}%', va='center', fontsize=10)
    ax.set_title('MAPE — Mean Absolute Percentage Error (Lower = Better)', fontweight='bold')
    ax.set_xlabel('MAPE (%)')
    ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    save_fig(fig, '03_mape_comparison.png')


### 📊 9.4 Actual vs Predicted & Residual Diagnostics (Best Model)

Loads the best-performing baseline model and generates:
1. **Actual vs Predicted Scatter Plot** — ideal predictions fall on $y = x$ line.
2. **Residual Error Distribution** — centered at zero indicates unbiased predictions.


In [ ]:
# ── 9.4 Actual vs Predicted & Residual Diagnostics ──
if baseline_results:
    best_name = df_results.iloc[0]['Model']
    print(f'🏆 Best Baseline: {best_name}')
    print(f'   Loading model for diagnostic plots...')

    # Generate predictions from best model
    try:
        # Determine which model file to load
        model_map = {
            'Linear Regression': 'baseline_linear_regression.joblib',
            'Ridge Regression': 'baseline_ridge.joblib',
            'Lasso Regression': 'baseline_lasso.joblib',
            'ElasticNet': 'baseline_elasticnet.joblib',
            'Random Forest': 'baseline_random_forest.joblib',
        }

        if best_name in model_map:
            best_model = joblib.load(MODELS_DIR / model_map[best_name])
            y_best = np.expm1(best_model.predict(X_val)).astype(np.float32)
            del best_model
        elif 'Seasonal' in best_name and snaive_vals is not None:
            y_best = snaive_vals.copy()
        elif 'Median' in best_name:
            y_best = np.full(len(y_val), float(np.median(y_train)), dtype=np.float32)
        else:
            y_best = np.full(len(y_val), float(np.mean(y_train)), dtype=np.float32)

        # Sample for scatter plot (avoid plotting millions of points)
        n_sample = min(5000, len(y_val))
        rng = np.random.RandomState(42)
        idx = rng.choice(len(y_val), size=n_sample, replace=False)

        y_actual_s  = y_val[idx]
        y_pred_s    = y_best[idx]
        residuals_s = y_actual_s - y_pred_s

        fig, axes = plt.subplots(1, 2, figsize=(15, 6))

        # Left: Actual vs Predicted
        axes[0].scatter(y_actual_s, y_pred_s, alpha=0.2, s=10, color='#2ecc71')
        lim = max(np.percentile(y_actual_s, 99), np.percentile(y_pred_s, 99))
        axes[0].plot([0, lim], [0, lim], 'r--', linewidth=2, label='Perfect (y=x)')
        axes[0].set_title(f'Actual vs Predicted — {best_name}', fontweight='bold')
        axes[0].set_xlabel('Actual Sales')
        axes[0].set_ylabel('Predicted Sales')
        axes[0].set_xlim(0, lim)
        axes[0].set_ylim(0, lim)
        axes[0].legend()
        axes[0].grid(True, alpha=0.3)

        # Right: Residual Distribution
        axes[1].hist(residuals_s, bins=60, color='#9b59b6', edgecolor='black', alpha=0.75)
        axes[1].axvline(0, color='red', linestyle='--', linewidth=2)
        axes[1].set_title('Residual Error Distribution (Actual − Predicted)', fontweight='bold')
        axes[1].set_xlabel('Residual Error')
        axes[1].set_ylabel('Frequency')
        axes[1].grid(True, alpha=0.3)

        plt.tight_layout()
        save_fig(fig, '04_actual_vs_predicted_residuals.png')

        del y_best, y_actual_s, y_pred_s, residuals_s, idx
        gc.collect()

    except Exception as e:
        print(f'⚠️ Diagnostic plot failed: {e}')


### 📊 9.5 Feature Correlation Heatmap

Visualizes Pearson correlation between the target and key features.


In [ ]:
# ── 9.5 Feature Correlation Heatmap ──
try:
    corr_cols = [TARGET_COL, 'sales_lag_1', 'sales_lag_7', 'sales_roll_mean_7',
                'sales_roll_mean_28', 'dcoilwtico', 'onpromotion']
    corr_cols = [c for c in corr_cols if c in df.columns]

    corr_sample = df[corr_cols].sample(n=min(500_000, len(df)), random_state=42)
    corr_mat = corr_sample.corr()
    del corr_sample

    fig, ax = plt.subplots(figsize=(10, 8))
    im = ax.imshow(corr_mat, cmap='coolwarm', vmin=-1, vmax=1)
    fig.colorbar(im, ax=ax)
    ax.set_xticks(np.arange(len(corr_cols)))
    ax.set_yticks(np.arange(len(corr_cols)))
    ax.set_xticklabels(corr_cols, rotation=45, ha='right')
    ax.set_yticklabels(corr_cols)
    ax.set_title('Feature Correlation Matrix with Target', fontweight='bold')
    for i in range(len(corr_cols)):
        for j in range(len(corr_cols)):
            ax.text(j, i, f'{corr_mat.iloc[i, j]:.2f}',
                    ha='center', va='center', color='black', fontsize=9)
    plt.tight_layout()
    save_fig(fig, '05_feature_correlation_heatmap.png')
    del corr_mat
except Exception as e:
    print(f'⚠️ Correlation heatmap failed: {e}')


## 🔟 Final Cleanup & Execution Summary


In [ ]:
# ============================================================
# 10. Final Cleanup & Execution Summary
# ============================================================
# Release all remaining large objects
for var in ['df', 'X_train', 'y_train', 'X_val', 'y_val', 'snaive_vals',
            'df_results', 'y_train_log']:
    if var in dir():
        exec(f'del {var}')
gc.collect()

total_time = time.time() - NOTEBOOK_START
log_ram('Final')

print(f'\n{"=" * 70}')
print('✅ NOTEBOOK 05 — BASELINE BENCHMARKING COMPLETE')
print(f'{"=" * 70}')
print(f'   Total Execution Time  : {total_time:.1f}s ({total_time/60:.1f} min)')
print(f'   Models Evaluated      : {len(baseline_results)}')
print(f'   Output Directory      : {OUTPUT_DIR}')
print(f'{"=" * 70}')


---

## ✅ Completion Checklist

| # | Requirement | Status |
|---|-------------|--------|
| 1 | Direct Parquet Load via `utils.load_feature_store()` | ✅ |
| 2 | Chronological Subset Selection (Last 10M rows, NO random sampling) | ✅ |
| 3 | Automatic Feature Detection & Leakage Prevention | ✅ |
| 4 | Strict Out-of-Time Chronological Train/Val Split | ✅ |
| 5 | 8 Baseline Models Trained & Evaluated | ✅ |
| 6 | RMSLE, RMSE, MAE, MAPE, R² Metrics Computed | ✅ |
| 7 | Training Time, Prediction Time, Model Size Measured | ✅ |
| 8 | `try/except` Error Handling Around Every Model | ✅ |
| 9 | RAM Monitoring After Every Major Operation | ✅ |
| 10 | `del` + `gc.collect()` Memory Cleanup After Every Model | ✅ |
| 11 | `plt.close(fig)` After Every Figure Save | ✅ |
| 12 | `baseline_results.csv` Exported | ✅ |
| 13 | `metrics.json` Exported | ✅ |
| 14 | `benchmark_summary.csv` Exported | ✅ |
| 15 | `feature_list.txt` Exported | ✅ |
| 16 | 5 Presentation-Grade Diagnostic Charts Saved | ✅ |
| 17 | Saved Models in `output/05_baseline_models/baseline_models/` | ✅ |

**➡️ Next:** Open `06_model_training.ipynb` for LightGBM, CatBoost, XGBoost training.
